# Pemodelan dan evaluasi

Notebook ini mendokumentasikan strategi yang dibandingkan dan kandidat lokal terbaik. Tidak ada pencarian eksperimen otomatis yang dijalankan saat notebook dibuka. Seluruh skor di sini adalah validasi lokal Macro F1, bukan skor Kaggle.

## Kesesuaian workflow dengan aturan

Skrip pembanding `scripts/train.py` membangun preprocessing dengan `ColumnTransformer` dan menggabungkannya dengan classifier dalam `Pipeline`. Sel berikut hanya memeriksa struktur pipeline; sel ini tidak melatih ulang model. Kandidat submission terbaik berasal dari OOF stacking XGBoost–LightGBM.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'train.py').exists() and (ROOT.parent / 'scripts' / 'train.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scripts.train import build_pipeline

pipeline = build_pipeline(
    {'depth': 6, 'learning_rate': 0.1, 'iterations': 300, 'l2_leaf_reg': 6},
    seed=2026,
)
assert isinstance(pipeline, Pipeline)
assert isinstance(pipeline.named_steps['preprocess'], ColumnTransformer)
print('Pipeline:', type(pipeline).__name__)
print('Preprocessing:', type(pipeline.named_steps['preprocess']).__name__)
print('Estimator:', type(pipeline.named_steps['model']).__name__)

## Perbandingan strategi multiclass

OvA mempelajari tiap kelas terhadap kelas lainnya. OvO memakai classifier pasangan; percobaan ini mengoreksi pasangan kelas yang berdekatan, yaitu 2/3 dan 3/4. Kandidat akhir menggabungkan beberapa model dasar melalui prediksi OOF dan kalibrasi kelas. Konfigurasi dan fitur tiap strategi berbeda, jadi tabel ini membandingkan hasil yang tersedia, bukan efek strategi secara terisolasi.

In [ ]:
import json
import pandas as pd

comparison = json.loads((ROOT / 'reports' / 'model_strategy_comparison.json').read_text(encoding='utf-8'))
results = pd.DataFrame(comparison['results'])
display(results[['strategy', 'macro_f1', 'notes']])
print('Split:', comparison['validation']['split'], '| seed:', comparison['validation']['seed'])

## Kandidat final yang dipilih

Kandidat lokal terbaik adalah `submissions/best_stack_087.csv`. Skor Macro F1 holdout **0,598551** berasal dari ensemble tiga XGBoost dan tiga LightGBM, logistic meta-model, lalu offset kelas yang dipilih dari prediksi OOF calibration. Grade 2 dan 3 masih paling sulit. Kalibrasi hanya dipilih dari calibration/OOF; evaluation holdout dipakai untuk mengukur hasil. Jangan menafsirkan skor ini sebagai skor Kaggle atau private leaderboard.

In [ ]:
summary = json.loads((ROOT / 'reports' / 'submission_best_stack_087.json').read_text(encoding='utf-8'))
print('Model:', summary['model'])
print('Macro F1 validasi lokal:', f"{summary['validation_macro_f1']:.6f}")
print('Baris CSV:', f"{summary['submission_rows']:,}")
print('Grade counts:', summary['grade_counts'])
print(summary['note'])